# 02 — Baseline Model

## Business case

Recap from `01_eda.ipynb`: the target (`target`, about 5.6% positive) marks whether a client is labelled as electricity/gas fraud. The operational goal is a ranked shortlist for a capacity-limited inspection team, so **PR-AUC (average precision)** is the primary metric, with precision@k and recall@k reported for the inspection budget.

## What this notebook does

It compares a `DummyClassifier` reference against a first real classifier: **logistic regression**. We use logistic regression rather than literal linear regression because the target is a 0/1 label, not a continuous quantity.

It loads the exact `df_train.csv` and `df_test.csv` created at the end of `01_eda.ipynb`. Both files contain `client_id`, `target`, and the 15 baseline inputs selected in the EDA: `invoice_count`, `active_days`, `mean_consumption`, `zero_consumption_rate`, `elec_share`, `mean_invoice_gap_days`, `backwards_index_rate`, `meter_count`, `mean_monthly_submission_index_delta`, `backward_submission_rate`, `large_mismatch_count`, `reconciliation_gap_abs_mean`, `client_catg`, `disrict`, and `region`.

Our five-fold cross-validation design remains inside `df_train`: it compares the models and creates out-of-fold predictions for capacity analysis. After that comparison, each model is fitted on all of `df_train` and evaluated on `df_test` as the final internal check.

The client split was created in `01_eda.ipynb` before the target-based EDA, so `df_test` was held aside while we made the feature decisions. The feature table is prepared for both splits, but the target-based comparisons use `df_train`. This is an internal labelled test split, not Kaggle's separate unlabelled test data.

## Imports and constants

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    average_precision_score,
    confusion_matrix,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", 100)

DATA_DIR = Path("data")
PROCESSED_DIR = DATA_DIR / "processed"
TRAIN_FEATURES_PATH = PROCESSED_DIR / "df_train.csv"
TEST_FEATURES_PATH = PROCESSED_DIR / "df_test.csv"

RSEED = 42  # same seed used to create the EDA train/test split
N_SPLITS = 5
TOP_K_SHARE = 0.10

for path in [TRAIN_FEATURES_PATH, TEST_FEATURES_PATH]:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {path}. Run 01_eda.ipynb with SAVE_FEATURES = True first."
        )

## Step 1 — Load the client-level train and test data

The EDA already created one row per client and selected the 15 baseline inputs. This notebook does not recreate invoice features.

In [ ]:
df_train = pd.read_csv(TRAIN_FEATURES_PATH)
df_test = pd.read_csv(TEST_FEATURES_PATH)

for frame in [df_train, df_test]:
    frame["client_id"] = frame["client_id"].astype("string")
    frame["target"] = frame["target"].astype("int8")

expected_columns = {
    "client_id", "target",
    "invoice_count", "active_days", "mean_consumption", "zero_consumption_rate",
    "elec_share", "mean_invoice_gap_days", "backwards_index_rate", "meter_count",
    "mean_monthly_submission_index_delta", "backward_submission_rate",
    "large_mismatch_count", "reconciliation_gap_abs_mean",
    "client_catg", "disrict", "region",
}
for name, frame in [("df_train", df_train), ("df_test", df_test)]:
    missing_columns = sorted(expected_columns - set(frame.columns))
    if missing_columns:
        raise ValueError(f"{name} is missing required columns: {missing_columns}")

assert set(df_train["client_id"]).isdisjoint(df_test["client_id"])

overview = pd.DataFrame({
    "clients": [len(df_train), len(df_test)],
    "unique_clients": [df_train["client_id"].nunique(), df_test["client_id"].nunique()],
    "fraud_rate": [df_train["target"].mean(), df_test["target"].mean()],
}, index=["df_train", "df_test"])
display(overview)
display(df_train.head())

## Step 2 — Preprocessing

The business-logic feature decisions already happened in `01_eda.ipynb`.

- **Excluded from the feature matrix:** `client_id` is a join key and `target` is the value to predict.
- **Numeric columns:** median-impute, then standard-scale. `mean_monthly_submission_index_delta` and `backward_submission_rate` are missing when a client has no usable positive-day meter transition. The imputer learns only from the training data.
- **Categorical columns:** one-hot encode `client_catg`, `disrict`, and `region`, with `handle_unknown="ignore"` so an unseen category does not crash the model.
- Preprocessing stays inside the model pipeline. During cross-validation it is fitted separately inside each training fold. For the final test it is fitted on all of `df_train`, never on `df_test`.

Raw dates, tenure, `has_negative_tenure`, `months_number` features, and investigation-related fields are not present because the EDA excluded them from the 15-input baseline.

In [ ]:
numeric_features = [
    "invoice_count",
    "active_days",
    "mean_consumption",
    "zero_consumption_rate",
    "elec_share",
    "mean_invoice_gap_days",
    "backwards_index_rate",
    "meter_count",
    "mean_monthly_submission_index_delta",
    "backward_submission_rate",
    "large_mismatch_count",
    "reconciliation_gap_abs_mean",
]
categorical_features = ["client_catg", "disrict", "region"]
model_features = numeric_features + categorical_features

X = df_train[model_features]
y = df_train["target"]
X_test = df_test[model_features]
y_test = df_test["target"]

print(f"{len(numeric_features)} numeric features, {len(categorical_features)} categorical features")
print(f"Cross-validation clients in df_train: {len(X):,}")
print(f"Final internal-test clients in df_test: {len(X_test):,}")

preprocessor = ColumnTransformer([
    ("numeric", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]), numeric_features),
    ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
])

## Step 3 — Dummy baseline vs. logistic regression

`class_weight="balanced"` gives the smaller fraud class more influence in logistic regression. It is a simple first response to the roughly 5.6% fraud rate.

First, both models use the same five-fold `StratifiedKFold` inside `df_train`. Our original model-comparison design: every validation fold is predicted by a model that did not fit on those clients.

Second, each pipeline is fitted on all of `df_train` and evaluated once on `df_test`. We report average precision (PR-AUC) as the primary metric and ROC-AUC as a secondary ranking metric.

In [ ]:
dummy_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", DummyClassifier(strategy="stratified", random_state=RSEED)),
])
logreg_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RSEED)),
])

cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RSEED)

comparison_rows = {}
for name, pipeline in [("dummy", dummy_pipeline), ("logistic_regression", logreg_pipeline)]:
    cv_results = cross_validate(
        pipeline, X, y, cv=cv, scoring=["average_precision", "roc_auc"]
    )
    comparison_rows[name] = {
        "average_precision_mean": cv_results["test_average_precision"].mean(),
        "average_precision_std": cv_results["test_average_precision"].std(),
        "roc_auc_mean": cv_results["test_roc_auc"].mean(),
        "roc_auc_std": cv_results["test_roc_auc"].std(),
    }

cv_model_comparison = pd.DataFrame(comparison_rows).T
print("Five-fold cross-validation results inside df_train:")
display(cv_model_comparison)

base_rate = y.mean()
print(f"Fraud rate in df_train: {base_rate:.4f}")

internal_test_rows = {}
for name, pipeline in [("dummy", dummy_pipeline), ("logistic_regression", logreg_pipeline)]:
    pipeline.fit(X, y)
    test_probability = pipeline.predict_proba(X_test)[:, 1]
    internal_test_rows[name] = {
        "average_precision": average_precision_score(y_test, test_probability),
        "roc_auc": roc_auc_score(y_test, test_probability),
    }

internal_test_comparison = pd.DataFrame(internal_test_rows).T
print("Final internal-test results on df_test:")
display(internal_test_comparison)

## Step 4 — Capacity-aware evaluation: precision@k / recall@k

Our original capacity analysis stays based on out-of-fold logistic-regression scores from `df_train`. Every client receives a prediction from a fold that did not train on that client. We rank those scores and inspect the top `TOP_K_SHARE`.

- Precision@k asks: of the clients selected for inspection, how many are fraud?
- Recall@k asks: of all fraud clients, how many did the selected group find?

In [ ]:
oof_proba = cross_val_predict(logreg_pipeline, X, y, cv=cv, method="predict_proba")[:, 1]

y_values = y.to_numpy()
k = int(np.ceil(TOP_K_SHARE * len(y_values)))
ranked_order = np.argsort(oof_proba)[::-1]
top_k_idx = ranked_order[:k]

precision_at_k = y_values[top_k_idx].mean()
recall_at_k = y_values[top_k_idx].sum() / y_values.sum()
implied_recall_at_k = precision_at_k * TOP_K_SHARE / base_rate  # cross-check via the identity
                                                                 # recall@k = precision@k * k / base_rate

capacity_summary = pd.DataFrame({
    "value": [k, precision_at_k, recall_at_k, implied_recall_at_k],
}, index=[
    f"clients inspected (top {TOP_K_SHARE:.0%})",
    "precision@k", "recall@k", "recall@k (via precision@k * k / base_rate, cross-check)",
])
display(capacity_summary)

threshold_at_k = oof_proba[ranked_order[k - 1]]
y_pred_at_k = (oof_proba >= threshold_at_k).astype(int)
cm = confusion_matrix(y_values, y_pred_at_k)
ConfusionMatrixDisplay(cm, display_labels=["not fraud", "fraud"]).plot(cmap="Blues")
plt.title(f"Confusion matrix at the top-{TOP_K_SHARE:.0%} cutoff")
plt.show()

### Precision-recall curve

This curve uses the same out-of-fold predictions from `df_train`. The horizontal line shows the fraud rate expected from an uninformative ranking.

In [ ]:
precisions, recalls, _ = precision_recall_curve(y_values, oof_proba)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(recalls, precisions, label="Logistic regression", color="#D62828")
ax.axhline(base_rate, linestyle="--", color="grey", label=f"Dummy floor ({base_rate:.1%})")
ax.scatter([recall_at_k], [precision_at_k], color="black", zorder=5,
           label=f"Top {TOP_K_SHARE:.0%} cutoff")
ax.set(xlabel="Recall", ylabel="Precision", title="Precision-recall curve (out-of-fold)")
ax.legend()
plt.tight_layout()
plt.show()

## Results

- **Dummy reference:** cross-validated PR-AUC **0.056** and ROC-AUC **0.500**, which is what we expect from an uninformative ranking.
- **Logistic regression:** cross-validated PR-AUC **0.168 ± 0.003** and ROC-AUC **0.793 ± 0.007** inside `df_train`. Logistic regression ranks fraud about three times better than the dummy PR-AUC floor.
- **Final internal test:** PR-AUC **0.177** and ROC-AUC **0.801** on `df_test`. This is close to the cross-validation result, so the baseline behaviour is reasonably consistent across the two saved splits.
- **Top 10% inspection capacity:** precision **20.45%** and recall **36.63%** from out-of-fold `df_train` predictions. If the team inspects 10,840 clients, about one in five selected clients is fraud and the list contains about 37% of all fraud clients in `df_train`.

Accuracy is not the headline result. With about 5.6% fraud, predicting “not fraud” for everyone would be highly accurate but would find no fraud.

### Next steps

- Use the out-of-fold predictions for initial false-positive and false-negative error analysis.
- Check performance by `client_catg`, `disrict`, `region`, and invoice-history length.
- Consider a more complex model only after understanding where logistic regression fails.
- Recreate the same 15 inputs for Kaggle's unlabelled client/invoice data and apply the fitted pipeline without refitting it.